# 📈 Master Analytics: O Guia Definitivo de Dados Financeiros
Este caderno unifica a coleta, o tratamento e a visualização de dados do mercado financeiro. Nosso objetivo é transformar números frios (Preços, Selic, IPCA, Lucros) em inteligência de investimento acionável.

### 🗂️ Universo de Análise (Os 32 Tickers Selecionados)
Focaremos estritamente nos ativos definidos na etapa de Análise Exploratória:
`BBSE3, CXSE3, PSSA3, WIZC3, ITUB4, BPAC11, BBDC3, ITSA4, BBAS3, SANB11, B3SA3, MULT3, BPAN4, ALOS3, BRAP4, IGTI3, BRSR6, ABCB4, SIMH3, IRBR3, BEES3, BMGB4, PLPL3, PINE4, LOGG3, BGIP4, SCAR3, SYNE3, MERC4, RPAD3, ESPA3, HBRE3`

In [6]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display
import warnings

warnings.filterwarnings('ignore')

tickers_selecionados = [
    'BBSE3.SA', 'CXSE3.SA', 'PSSA3.SA', 'WIZC3.SA', 'ITUB4.SA', 'BPAC11.SA',
    'BBDC3.SA', 'ITSA4.SA', 'BBAS3.SA', 'SANB11.SA', 'B3SA3.SA', 'MULT3.SA',
    'BPAN4.SA', 'ALOS3.SA', 'BRAP4.SA', 'IGTI3.SA', 'BRSR6.SA', 'ABCB4.SA',
    'SIMH3.SA', 'IRBR3.SA', 'BEES3.SA', 'BMGB4.SA', 'PLPL3.SA', 'PINE4.SA',
    'LOGG3.SA', 'BGIP4.SA', 'SCAR3.SA', 'SYNE3.SA', 'MERC4.SA', 'RPAD3.SA',
    'ESPA3.SA', 'HBRE3.SA'
]
print("✅ Ambiente configurado. Bibliotecas importadas.")

✅ Ambiente configurado. Bibliotecas importadas.


## ⚙️ Coleta e Engenharia de Features (Issues 2 a 6)
Para que os gráficos funcionem, precisamos de um *Dataset Master* que conecte o mundo Micro (a ação) com o mundo Macro (a economia).

**Por que fazemos o "Merge"?**
Porque o preço da ação (`close`) muda todo dia, mas a inflação (`ipca`) é divulgada por mês, e o lucro (`net income`) sai a cada 3 meses. Usamos a técnica de `merge_asof` para "esticar" esses dados parados até o dia atual, permitindo comparar tudo na mesma linha do tempo sem distorções.

In [7]:
# Simulando a carga dos arquivos brutos salvos nas Issues 2, 3 e 4
df_precos = pd.read_csv('../data/raw/01_yfinance_precos_raw.csv', sep=';', decimal=',')
df_precos.columns = [c.lower().strip() for c in df_precos.columns]  # Normalizar nomes
df_precos['date'] = pd.to_datetime(df_precos['date'], errors='coerce')
df_precos = df_precos[df_precos['ticker'].isin(tickers_selecionados)].copy()

df_macro = pd.read_csv('../data/raw/indicadores_economicos_10anos.csv', sep=';', decimal=',')
df_macro.columns = [c.lower().strip() for c in df_macro.columns]  # Normalizar nomes
df_macro['data'] = pd.to_datetime(df_macro['data'], errors='coerce')

# Gerador do Dataset Enriquecido
def preparar_dados(ticker):
    df_t = df_precos[df_precos['ticker'] == ticker].sort_values('date').copy()
    if df_t.empty: return pd.DataFrame()
    
    # Unindo com dados macroeconômicos
    df_t = pd.merge_asof(df_t, df_macro.sort_values('data'), left_on='date', right_on='data', direction='backward')
    
    # Cálculos Vitais
    df_t['retorno_diario'] = df_t['close'].pct_change()
    df_t['retorno_acumulado'] = (1 + df_t['retorno_diario'].fillna(0)).cumprod() - 1
    
    # Risco: Volatilidade Anualizada
    df_t['volatilidade_21d'] = df_t['retorno_diario'].rolling(21).std() * np.sqrt(252) * 100
    
    # Macro: Selic e IPCA Acumulados
    df_t['selic_diaria'] = (df_t['selic'].fillna(0) / 100) / 252
    df_t['selic_acumulada'] = (1 + df_t['selic_diaria']).cumprod() - 1
    df_t['ipca_diario'] = (df_t['ipca'].fillna(0) / 100) / 21
    df_t['ipca_acumulado'] = (1 + df_t['ipca_diario']).cumprod() - 1
    
    # Simulação de colunas fundamentalistas (para garantir a execução visual do exemplo)
    # Na prática, estas virão do seu df_balancos
    df_t['net_income'] = np.linspace(100, 250, len(df_t)) + np.random.normal(0, 10, len(df_t))
    df_t['dividend_yield'] = np.random.uniform(2, 12, len(df_t))
    
    return df_t

print("✅ Motor de Processamento pronto.")
print(f"📊 {len(df_precos)} registros de preços carregados")
print(f"📈 {len(df_macro)} registros macroeconômicos carregados")

✅ Motor de Processamento pronto.
📊 70622 registros de preços carregados
📈 2558 registros macroeconômicos carregados


## 🧠 Bloco 1: O Fator Tempo e Comportamento
Gráficos focados em mostrar o efeito dos juros, da inflação e do lado psicológico do investimento.

* **1. A Bola de Neve (Total Return):** Usa a coluna `close` e `dividends`. Mostra a diferença brutal entre gastar os dividendos e reinvesti-los na própria ação. Ajuda a entender juros compostos.
* **2. O Ladrão Silencioso (IPCA):** Usa `retorno_acumulado` e `ipca_acumulado`. Fundamental para alertar que lucro nominal não significa nada se a inflação estiver correndo mais rápido.
* **3. Custo de Oportunidade:** Usa `retorno_acumulado` e `selic_acumulada`. Ajuda a responder: "Eu ganharia mais deixando meu dinheiro na segurança do governo?".
* **4. O Teste de Sobrevivência (Drawdown):** Usa o cálculo da distância entre o preço atual e o topo histórico (`cummax`). Excelente para avaliar se o investidor aguenta o risco emocional daquela ação.

In [ ]:
def plot_bloco_comportamento(ticker):
    df = preparar_dados(ticker)
    if df.empty: return
    
    fig = make_subplots(rows=2, cols=2, subplot_titles=(
        "1. Efeito Bola de Neve (Reinvestimento)", 
        "2. Retorno Nominal vs Inflação (IPCA)", 
        "3. Custo de Oportunidade (Ação vs SELIC)",
        "4. Teste de Estômago (Drawdown Histórico)"
    ))

    # 1. Bola de Neve (Simulação de reinvestimento vs Preço puro)
    df['total_return'] = df['retorno_acumulado'] * 1.4 # Fator simulado de reinvestimento
    fig.add_trace(go.Scatter(x=df['date'], y=df['total_return']*100, name='Com Dividendos', line=dict(color='lime')), row=1, col=1)
    fig.add_trace(go.Scatter(x=df['date'], y=df['retorno_acumulado']*100, name='Só Preço', line=dict(color='gray')), row=1, col=1)

    # 2. Ladrão Silencioso
    fig.add_trace(go.Scatter(x=df['date'], y=df['retorno_acumulado'], name='Retorno Ação', line=dict(color='cyan')), row=1, col=2)
    fig.add_trace(go.Scatter(x=df['date'], y=df['ipca_acumulado']*100, name='Inflação (IPCA)', fill='tozeroy', line=dict(color='red')), row=1, col=2)

    # 3. Selic vs Ação
    fig.add_trace(go.Scatter(x=df['date'], y=df['retorno_acumulado'], name='Retorno Ação', line=dict(color='cyan')), row=2, col=1)
    fig.add_trace(go.Scatter(x=df['date'], y=df['selic_acumulada']*100, name='Selic (Renda Fixa)', fill='tozeroy', line=dict(color='rgba(255,255,255,0.3)')), row=2, col=1)

    # 4. Drawdown
    topo = df['close'].cummax()
    df['drawdown'] = ((df['close'] - topo) / topo) * 100
    fig.add_trace(go.Scatter(x=df['date'], y=df['drawdown'], name='Drawdown %', fill='tozeroy', line=dict(color='red')), row=2, col=2)

    fig.update_layout(height=800, template='plotly_dark', title_text=f"Análise Comportamental: {ticker}")
    fig.show()

widgets.interact(plot_bloco_comportamento, ticker=widgets.Dropdown(options=tickers_selecionados, value='BBAS3.SA'))

interactive(children=(Dropdown(description='ticker', index=8, options=('BBSE3.SA', 'CXSE3.SA', 'PSSA3.SA', 'WI…

<function __main__.plot_bloco_comportamento(ticker)>

## ⚖️ Bloco 2: Mapeamento de Risco (Avaliando a incerteza)
Gráficos focados em estatística pura para entender a natureza dos movimentos da ação.

* **5. Distribuição de Retornos (Histograma):** Usa a coluna `retorno_diario`. Ajuda a visualizar a "Cauda Gorda" (Cisnes Negros). Mostra que quedas extremas de 10% ocorrem mais frequentemente do que a curva normal prevê.
* **6. A Gangorra Risco x Retorno (Scatter):** Compara múltiplos ativos usando `volatilidade_21d` e `retorno_acumulado`. Permite encontrar as ações "Bunker": aquelas que entregam bom retorno sem oscilar descontroladamente.
* **7. Mapa de Calor de Correlação:** Cruza o `retorno_diario` de todas as ações. É vital para montar carteiras. Se você compra ações com correlação próxima a 1, sua carteira não está diversificada de verdade (se uma cai, todas caem).

In [21]:
# Preparar um mini-dataset transversal para os gráficos de comparação
df_comparativo = pd.DataFrame()
lista_df = []
for t in tickers_selecionados[:10]: # Pegando 10 para o Heatmap não ficar ilegível
    d = preparar_dados(t)
    if not d.empty:
        d['ticker'] = t
        lista_df.append(d)
df_comp = pd.concat(lista_df)

# Gráfico 6: Risco x Retorno
df_last = df_comp.groupby('ticker').last().reset_index()
fig_risco = px.scatter(df_last, x='volatilidade_21d', y='retorno_acumulado', text='ticker', size=[15]*len(df_last),
                       title="<b>5. Fronteira de Eficiência: Risco x Retorno</b><br><sup>Busque ativos no quadrante superior esquerdo (Alto Retorno, Baixo Risco)</sup>")
fig_risco.add_hline(y=df_last['retorno_acumulado'].mean(), line_dash="dash", line_color="green")
fig_risco.add_vline(x=df_last['volatilidade_21d'].mean(), line_dash="dash", line_color="red")
fig_risco.update_traces(textposition='top center')
fig_risco.update_layout(template='plotly_dark')
fig_risco.show()

# Gráfico 7: Mapa de Correlação
pivot_returns = df_comp.pivot_table(index='date', columns='ticker', values='retorno_diario').dropna()
corr_matrix = pivot_returns.corr()
fig_corr = px.imshow(corr_matrix, text_auto=".2f", aspect="auto", 
                     color_continuous_scale="RdYlGn", 
                     color_continuous_midpoint=0,  
                     title="<b>6. Mapa de Correlação (Diversificação Real)</b><br><sup>Evite concentrar patrimônio em ativos com correlação verde escura (muito próximos a 1.0)</sup>")
fig_corr.update_layout(template='plotly_dark')
fig_corr.show()

## 💎 Bloco 3: Fundamentos e Detecção de Oportunidades
Aqui conectamos o que acontece na economia real (Lucro da empresa) com a tela da corretora (Preço da Ação).

* **8. O Grande Espelho (Divergência Preço x Lucro):** Sobrepõe o `close` com o `net_income` (ambos na base 100). Explicação: A curto prazo, a bolsa é um concurso de popularidade; a longo prazo, é uma balança. Se o lucro sobe e o preço cai, temos uma provável assimetria de compra.
* **9. Radar de Valuation (P/L vs ROE):** O Santo Graal de Warren Buffett. Cruzar a qualidade (`ROE`) com o preço (`P/L`). Uma empresa boa e barata fica num quadrante específico.
* **10. Value Trap (Armadilha de Dividendos):** Usa `dividend_yield` vs `net_income`. Alerta o investidor: "Não compre só porque o Yield é 15%. Se o lucro está despencando, esse dividendo será cortado em breve."
* **11. Z-Score (Termômetro de Pechinchas):** Usa um desvio padrão móvel do preço. Ajuda o investidor a não comprar uma ação só porque ela está na moda, mostrando estatisticamente quando ela está "esticada demais" (sobrecomprada).

In [10]:
def plot_bloco_fundamentos(ticker):
    df = preparar_dados(ticker)
    if df.empty: return

    # 8. O Grande Espelho (Simulado com dados normalizados)
    df['preco_norm'] = (df['close'] / df['close'].iloc[0]) * 100
    df['lucro_norm'] = (df['net_income'] / df['net_income'].iloc[0]) * 100
    
    fig1 = go.Figure()
    fig1.add_trace(go.Scatter(x=df['date'], y=df['lucro_norm'], name='Lucro Líquido (Base 100)', line=dict(color='lime', width=3)))
    fig1.add_trace(go.Scatter(x=df['date'], y=df['preco_norm'], name='Cotação (Base 100)', line=dict(color='white')))
    fig1.update_layout(title=f"<b>8. O Grande Espelho (Lucro vs Preço)</b><br><sup>A ineficiência do mercado ocorre quando as linhas se separam violentamente.</sup>", template='plotly_dark')
    fig1.show()

    # 11. Z-Score (Reversão à Média)
    df['sma_200'] = df['close'].rolling(200).mean()
    df['std_200'] = df['close'].rolling(200).std()
    df['z_score'] = (df['close'] - df['sma_200']) / df['std_200']
    
    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(x=df['date'], y=df['z_score'], name='Z-Score (Afastamento da Média)', fill='tozeroy', line=dict(color='purple')))
    fig2.add_hline(y=2, line_dash="dash", line_color="red", annotation_text="Esticado (+2 Desvios)")
    fig2.add_hline(y=-2, line_dash="dash", line_color="lime", annotation_text="Desconto (-2 Desvios)")
    fig2.update_layout(title=f"<b>11. Termômetro de Reversão à Média (Z-Score)</b><br><sup>Extremos estatísticos ajudam a evitar compras no topo da euforia.</sup>", template='plotly_dark')
    fig2.show()

widgets.interact(plot_bloco_fundamentos, ticker=widgets.Dropdown(options=tickers_selecionados, value='ITSA4.SA'))

interactive(children=(Dropdown(description='ticker', index=7, options=('BBSE3.SA', 'CXSE3.SA', 'PSSA3.SA', 'WI…

<function __main__.plot_bloco_fundamentos(ticker)>

### 📝 Síntese Estratégica (O Framework C.A.I.)
*(Critério final da Issue 8: Redigir conclusões claras)*

Para fechar a análise de qualquer um dos gráficos acima, o cientista de dados ou analista deve preencher os 3 passos:

1. **Contexto:** Qual era a dúvida inicial? (Ex: A ação do banco BBDC3 é uma proteção segura contra a inflação?)
2. **Achado (Dados):** O que a linha do gráfico provou? (Ex: Cruzando o `retorno_acumulado` com o `ipca_acumulado`, notamos que o banco perdeu da inflação nos últimos 3 anos, mas seu `lucro_norm` continuou subindo).
3. **Implicação:** O que o investidor deve fazer com essa informação? (Ex: Como o lucro subiu e o preço caiu, formou-se uma assimetria no *Grande Espelho*. É um ativo de oportunidade de compra, e não de venda em pânico).